In [44]:
import cv2
import numpy as np


In [45]:
img1 = cv2.imread('./images/image1.png')
img2 = cv2.imread('./images/image2.png')


In [46]:
# Initialize SIFT detector
sift = cv2.SIFT_create()

# Detect keypoints and descriptors
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# Use BFMatcher to find matches
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches = bf.match(des1, des2)

# Sort matches by distance
matches = sorted(matches, key=lambda x: x.distance)
# matches

In [47]:
img_matches = cv2.drawMatches(
    img1, kp1, img2, kp2, matches[:50], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
cv2.imshow("local test", img_matches)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [48]:
# Extract location of good matches
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# Compute homography
H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
H


array([[ 1.00000000e+00,  3.73063190e-16, -1.46800398e-14],
       [ 2.26173346e-17,  1.00000000e+00,  8.50171106e-14],
       [ 1.61733217e-19,  7.28781304e-19,  1.00000000e+00]])

In [49]:
# Get the dimensions of the images
h1, w1 = img1.shape[:2]
h2, w2 = img2.shape[:2]

# Get the canvas dimesions
pts = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
dst = cv2.perspectiveTransform(pts, H)
img2_warped = cv2.warpPerspective(img2, H, (w1 + w2, h1))

# Place the first image on the canvas
img2_warped[0:h1, 0:w1] = img1

In [51]:
# Simple blending technique
result = img2_warped

cv2.imshow('Result', result)
cv2.waitKey(0)
cv2.destroyAllWindows()
